In [1]:
import ee
import xarray
import rioxarray
import rasterio
from rasterio.transform import from_origin
import threading
import warnings

In [2]:
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [3]:
def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)



In [4]:
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU")

#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], 
          #['UBlue', 'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']).filterBounds(LTBMU.geometry())

ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())

ic_xr = xarray.open_dataset(ic, engine = "ee", crs='EPSG:3310', scale = 30)

In [ ]:
print(ic_xr)
red_band = ic_xr['SR_B4']
print(red_band)

In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [13]:
print(da)

<xarray.Dataset>
Dimensions:      (band: 3, x: 791, y: 718)
Coordinates:
  * band         (band) int32 1 2 3
  * x            (x) float64 1.021e+05 1.024e+05 ... 3.389e+05 3.392e+05
  * y            (y) float64 2.827e+06 2.826e+06 ... 2.612e+06 2.612e+06
    spatial_ref  int32 ...
Data variables:
    band_data    (band, y, x) float32 ...


In [12]:
print(ic_xr.coarsen.__doc__)


        Coarsen object for Datasets.

        Parameters
        ----------
        dim : mapping of hashable to int, optional
            Mapping from the dimension name to the window size.
        boundary : {"exact", "trim", "pad"}, default: "exact"
            If 'exact', a ValueError will be raised if dimension size is not a
            multiple of the window size. If 'trim', the excess entries are
            dropped. If 'pad', NA will be padded.
        side : {"left", "right"} or mapping of str to {"left", "right"}, default: "left"
        coord_func : str or mapping of hashable to str, default: "mean"
            function (name) that is applied to the coordinates,
            or a mapping from coordinate name to function (name).

        Returns
        -------
        core.rolling.DatasetCoarsen

        See Also
        --------
        core.rolling.DatasetCoarsen
        DataArray.coarsen

        :ref:`reshape.coarsen`
            User guide describing :py:func:`~xarray.D

In [ ]:
factor = 100 // 30
resample = ic_xr.coarsen(X=factor, Y=factor).mean()

In [ ]:
warnings.filterwarnings("ignore")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU")

#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], 
          #['UBlue', 'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']).filterBounds(LTBMU.geometry())

ic_landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())
ic_dem = ee.ImageCollection("USGS/3DEP/1m").filterBounds(LTBMU.geometry())
ic_xr = xarray.open_mfdataset([ic_landsat, ic_dem], engine = "ee", crs='EPSG:3310', scale = 30)



In [ ]:
ds = xarray.open_mfdataset(['ee://ECMWF/ERA5_LAND/HOURLY', 'ee://NASA/GDDP-CMIP6'],
                           engine='ee', crs='EPSG:4326', scale=0.25)

In [5]:
red_band = ic_xr['SR_B4'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
nir_band = ic_xr['SR_B5'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
ndvi = (nir_band - red_band) / (nir_band + red_band)
ndvi_transpose = ndvi.transpose('Y', 'X').rename({"X": "x", "Y": "y"})

In [ ]:
print(ndvi_transpose.rio.to_raster.__doc__)

In [ ]:
print(ic_xr['SR_B5'].isel(time=0).chunk.__doc__)

In [ ]:
print(ndvi_transpose.to_netcdf.__doc__)

In [6]:
ndvi_transpose.rio.to_raster(raster_path="ndvi.tif", windowed=True, tiled=True, lock=threading.Lock())

c:\Users\knigh\anaconda3\envs\ee\Lib\site-packages\dask\core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


In [30]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate('1992-10-05', '1993-03-31')
leg1 = ee.Geometry.Rectangle(113.33, -43.63, 153.56, -10.66)
ds = xarray.open_dataset(
    ic,
    engine='ee',
    projection=ic.first().select(0).projection(),
    geometry=leg1
)

In [ ]:
ds_renamed = ds.rename({'lon': 'x', 'lat': 'y'})
ds_renamed_transpose = ds_renamed.transpose('time', 'y', 'x')
ds_renamed_transpose.isel(time=0).rio.to_raster("hourly.tif")
#ds_renamed_transpose.rio.to_raster(output_raster_path, driver='GTiff', crs='EPSG:4326', compress='lzw')

In [ ]:
'''ds_img = ds.sel(time='1992-10-05T00:00:00.000000000')
print(ds_img.coords)
lat_range = slice(-10, -20.0)  # Replace with your desired latitude range
lon_range = slice(120, 130)  # Replace with your desired longitude range

# Use .sel to filter the dataset
tiled_img = ds_img.sel(lat=lat_range, lon=lon_range)
print(tiled_img.coords)
# Print the filtered dataset
#print(tiled_img)'''